In [ ]:
import math


# --------------------------
# Erlang B
# --------------------------
def erlang_b(servers, intensity):
    if servers <= 0 or intensity < 0:
        return 0

    b = 1
    for i in range(1, int(servers) + 1):
        b = (intensity * b) / (i + intensity * b)

    return max(0, min(b, 1))


# --------------------------
# Erlang C
# --------------------------
def erlang_c(agents, intensity):

    if agents <= intensity:
        return 1

    b = erlang_b(agents, intensity)

    c = b / (
        ((intensity / agents) * b)
        + (1 - (intensity / agents))
    )

    return max(0, min(c, 1))


# --------------------------
# Utilisation
# --------------------------
def utilisation(agents, calls_per_hour, aht):

    traffic = (calls_per_hour * aht) / 3600

    return traffic / agents


# --------------------------
# SLA
# --------------------------
def sla(agents, service_time, calls_per_hour, aht):

    traffic = (calls_per_hour * aht) / 3600

    if agents <= traffic:
        return 0

    c = erlang_c(agents, traffic)

    sla_result = 1 - (
        c *
        math.exp(
            (traffic - agents)
            * service_time
            / aht
        )
    )

    return round(
        max(0, min(sla_result, 1)) * 100,
        2
    )


# --------------------------
# Agents Required
# --------------------------
def agents_required(
        target_sla,
        service_time,
        calls_per_hour,
        aht
):

    traffic = (
        calls_per_hour *
        aht
    ) / 3600

    agents = max(
        1,
        math.ceil(traffic)
    )

    while True:

        current_sla = sla(
            agents,
            service_time,
            calls_per_hour,
            aht
        ) / 100

        if current_sla >= target_sla:
            return agents

        agents += 1


# --------------------------
# Example
# --------------------------

calls = 18 # @param {type:"integer"}
aht = 519 # @param {type:"integer"}
target_sla = 0.80 # @param {type:"number"}
service_time = 30 # @param {type:"integer"}

required = agents_required(
    target_sla,
    service_time,
    calls,
    aht
)

print(f"Agents Required: {required}")

print(
    f"SLA Achieved: "
    f"{sla(required, service_time, calls, aht)}%"
)

print(
    f"Utilisation: "
    f"{utilisation(required, calls, aht):.2%}"
)

In [ ]:
  import matplotlib.pyplot as plt
import numpy as np

# Define a range for the number of agents around the 'required' value
agents_range_min = max(1, required - 10) # Ensure it doesn't go below 1
agents_range_max = required + 10
agent_options = np.arange(agents_range_min, agents_range_max + 1)

sla_values = []
for num_agents in agent_options:
    current_sla = sla(num_agents, service_time, calls, aht)
    sla_values.append(current_sla)

# Plotting the results
plt.figure(figsize=(12, 6))
plt.plot(agent_options, sla_values, marker='o', linestyle='-', color='skyblue', label='Calculated SLA')

# Highlight the target SLA and the required agents
plt.axhline(y=target_sla * 100, color='r', linestyle='--', label=f'Target SLA ({target_sla*100:.0f}%)')
plt.axvline(x=required, color='green', linestyle=':', label=f'Required Agents ({required})')

plt.title('SLA vs. Number of Agents')
plt.xlabel('Number of Agents')
plt.ylabel('SLA (%)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(agent_options)
plt.legend()
plt.tight_layout()
plt.show()